# Reading Real-Time Streaming Messages from Kafka Topic
* Topic: Reading Real-Time Streaming Messages from Kafka Topic
* Author: Oindrila Chakraborty

# Payload Structure
* Below is the sample <b>JSON</b> payload structure -

In [0]:
{
    "event_id": "1895604u-v56j-2504-a6b7-12hgtm0941",
    "event_offset": 10001,
    "event_publisher": "hr_system",
    "employee_id": "E1001",
    "data": {
        "assignments": [
            {
                "assignment_id": "A001",
                "start_from": "1st Jan 2024",
                "end_at": "30th Jun 2024",
                "status": "COMPLETED"
            },
            {
                "assignment_id": "A002",
                "start_from": "1st Jul 2024",
                "end_at": "31st Dec 2024",
                "status": "COMPLETED"
            },
            {
                "assignment_id": "A003",
                "start_from": "1st Jan 2025",
                "end_at": "31st Jul 2025",
                "status": "COMPLETED"
            },
            {
                "assignment_id": "A004",
                "start_from": "1st Aug 2025",
                "end_at": "31st Jan 2026",
                "status": "COMPLETED"
            },
            {
                "assignment_id": "A005",
                "start_from": "1st Feb 2026",
                "end_at": "30th Jun 2026",
                "status": "COMPLETED"
            },
            {
                "assignment_id": "A006",
                "start_from": "1st Jul 2026",
                "end_to": null,
                "status": "ACTIVE"
            }
        ]
    },
    "event_time": "2026-07-02 05:02:34.560325"
}

# How to Use Kafka as a Streaming Source to Read Data from Kafka Topic in Databricks Using Spark Structured Streaming Code?
* There is no need to upload or enable an <b>External Spark JAR File</b> to read <b>Kafka Streaming</b> data on either the <b>Classic Compute</b> or the <b>Serverless Compute</b>.
* The <b>Databricks Platform</b> natively bundles the <b>spark-sql-kafka</b> integration out of the box.
<br>This means it is possible to invoke <b>spark.readStream.format("kafka")</b> instantly via the <b>SparkSession</b> in a <b>Notebook</b>.

# How to Decide from Which Offset to Read from Kafka?
* When reading messages from <b>Apache Kafka</b> using <b>Databricks</b> via either <b>Spark Structured Streaming</b>, or <b>Batch Queries</b>, the <b>startingOffsets</b> option dictates from where the <b>Consumer</b>, i.e, the <b>Databricks</b> begins reading the messages.
* Below are the three types of values that can be passed to the <b>startingOffsets</b> option as <b>.option("", <value>)</b> to denote the starting point (<b>Offset</b>) from where the <b>Consumer</b>, i.e, the <b>Databricks</b>:
    * <b>1</b>. <b>latest</b>: This is the default value for <b>Streaming</b>, and, only applicable for <b>Streaming</b>.
    <br>In this case, <b>Consumer</b>, i.e, the <b>Databricks</b> ignores all historical messages in the <b>Kafka Topic</b>, and, only reads the messages that arrive after the <b>Streaming</b> starts.
        * The important thing to remember is that this value is invalid and not allowed for <b>Batch Queries</b>.
    * <b>2</b>. <b>earliest</b>: In this case, <b>Consumer</b>, i.e, the <b>Databricks</b> reads all available historical messages from the beginning of the <b>Kafka Topic Partitions</b> before processing new messages.
    * <b>3</b>. <b>JSON String (Per-Partition Offsets)</b>: In this case, a <b>Structured JSON String</b> specifies the precise starting <b>Offsets</b> for individual <b>Partitions</b> in one, or, multiple <b>Kafka Topics</b>.
    <br>In this <b>JSON format</b>, <b>-2</b> is used to represent the <b>earliest Offset</b>, and, <b>-1</b> is used to represent the <b>latest Offset</b>.
    <br>Example of a <b>Structured JSON String</b> -
        * <b>"""{"topic_name":{"0":1234, "1":-1, "2":-2}}"""</b>

# Read Kafka Message from a Topic Using Spark Structured Streaming With Offset Option
* When no <b>Checkpoint</b> exists while reading the messages from a <b>Kafka Topic</b>, i.e., when the <b>Kafka Topic</b> is read for the first time, via <b>Spark Structured Streaming</b>, then setting the <b>startingOffsets</b> option to <b>earliest</b> forces the <b>Stream</b> to read the <b>Kafka Topic</b> from the very first available message.

In [0]:
df = (
    spark.readStream
            .format("kafka")
            .option("kafka.bootstrap.servers", "<BROKER_IP>:<PORT>")
            .option("subscribe", "topic_name")
            .option("startingOffsets", "earliest") # Begins Processing from the Oldest Available Message in the Topic
            .load()
)

# Read Kafka Message from a Topic Using Batch Query With Offset Option
* While reading the messages from a <b>Kafka Topic</b> via a <b>One-Time Batch Load Query</b>, the <b>startingOffsets</b> option must be combined with <b>endingOffsets</b> option.
* The <b>startingOffsets</b> option must be set to <b>earliest</b> so that the <b>Consumer</b>, i.e, the <b>Databricks</b> can read the <b>Kafka Topic</b> from the very first available message.
* The <b>endingOffsets</b> option must be set to <b>latest</b> so that the <b>Consumer</b>, i.e, the <b>Databricks</b> can read the <b>Kafka Topic</b> till the very last available message.

In [0]:
df = (
    spark.readStream
            .format("kafka")
            .option("kafka.bootstrap.servers", "<BROKER_IP>:<PORT>")
            .option("subscribe", "topic_name")
            .option("startingOffsets", "earliest") # Begins Processing from the Oldest Available Message in the Topic
            .option("endingOffsets", "latest") # Processes Till the Last Message in the Topic
            .load()
)

# Read Kafka Message from a Topic Using Granular Control via Partition JSON With Offset Option
* While reading the messages from a <b>Kafka Topic</b>, it is possible to configure a <b>Hybrid Start Scenario</b> across multiple <b>Partitions</b> of that <b>Kafka Topic</b>, which is needed for a precise <b>Recovery</b>, or, <b>Backfill</b> strategy.
* In this below example, the configuration holds the granular control regarding from which <b>Offset</b> positions, each of the <b>Partitions</b> would read the messages -
    * The <b>Consumer</b>, i.e, the <b>Databricks</b> would start to read messages in <b>Partition 0</b> starting from the <b>Offset #45001</b>.
    * The <b>Consumer</b>, i.e, the <b>Databricks</b> would start to read messages in <b>Partition 1</b> starting from the absolute oldest message.
    * The <b>Consumer</b>, i.e, the <b>Databricks</b> would only read newly arriving messages in <b>Partition 2</b>.

In [0]:
partition_json = """{"my_topic":{"0":45001,"1":-2,"2":-1}}"""

df = (
    spark.readStream
            .format("kafka")
            .option("kafka.bootstrap.servers", "<BROKER_IP>:<PORT>")
            .option("subscribe", "topic_name")
            .option("startingOffsets", partition_json)
            .load()
)

# When Checkpoint Already Keeps Track of the Offset While Reading Message from Kafka Topic, Then Why startingOffsets Option is Used?
* The <b>startingOffsets</b> option is only applicable when the <b>Consumer</b>, i.e, the <b>Databricks</b> starts to read the messages from a <b>Kafka Topic</b> for the very first time via <b>Spark Structured Streaming</b>.
* If the <b>Streaming Job</b> restarts and points to an existing <b>Checkpoint</b> directory, then <b>Spark</b> ignores the value provided in the <b>startingOffsets</b> option, and, safely resumes from where it last left off based on the metadata stored in the <b>Checkpoint</b> directory.

# Subscribe to Kafka Topic and Read Streaming Message Data

## Batch Format Code

* To read data from <b>Kafka</b>, the following lines of code are used -
    * <b>.format ("kafka")</b> specifies the data source as <b>Kafka</b>.
    * <b>.option("kafka.bootstrap.servers", "host1:port1,host2:port2")</b> specifies the <b>Kafka Bootstrap Servers</b>.
    * <b>.option("subscribe", "topic_name")</b> specifies the <b>Kafka Topic</b> to subscribe to.
    * <b>.option("startingOffsets", "earliest")</b> specifies the starting offset to be used when a query without an <b>offset</b> is started.
    * <b>.option("endingOffsets", "latest")</b> specifies the ending offset to be used when a query without an <b>offset</b> is started.
    * <b>.load()</b> loads the data into a <b>DataFrame</b>.

In [0]:
df = (
    spark.read
            .format("kafka")
            .option("kafka.bootstrap.servers", "<BROKER_IP_1>:<PORT_1>, <BROKER_IP_2>:<PORT_2>")
            .option("subscribe", "topic_name")
            .option("startingOffsets", "earliest") # Begins Processing from the Oldest Available Message in the Topic
            .option("endingOffsets", "latest") # Processes Till the Last Message in the Topic
            .load()
)

# Issue With Schema Inference While Reading Messages from Kafka Topic Using Spark Structured Streaming in Databricks
* By default, <b>Spark Structured Streaming</b> requires an explicitly defined schema for reading messages from <b>Kafka Topic</b> to guarantee that the stream is stable and consistent.
* Although, setting the configuration <b>"spark.sql.streaming.schemaInference"</b> to <b>"true"</b> in the current <b>SparkSession</b> forces <b>Spark</b> to infer, or, read the <b>Schema</b> from the <b>Streaming</b> data itself at run time, but, this configuration does not work on <b>Message Queues</b>, like - <b>Kafka</b>.

# Solution of the Issue With Schema Inference While Reading Messages from Kafka Topic Using Spark Structured Streaming in Databricks

## Programmatically Infer Schema via Batch Samples
* For the <b>Messaging Queue Streaming</b> source, like - <b>Kafka</b>, <b>Spark</b> cannot infer <b>Schemas</b> on the fly out of <b>Raw Binary</b> payloads.
* In this case, the following steps should be used -
    * <b>1</b>. <b>The Sample Batch</b>: <b>Spark</b> should be connected to live <b>Kafka Topic</b> using the <b>Spark</b> function <b>spark.read</b> for <b>Batch</b>, instead of the <b>Spark</b> function <b>spark.readStream</b> for <b>Stream</b>.
    <br>It pulls a small sample, e.g., the last 100 messages.
    * <b>2</b>. <b>The Schema Extraction</b>: Those sample messages should be cast to <b>Strings</b> and one valid message out of those messages should be passed to the built-in <b>Spark</b> function <b>schema_of_json()</b>.
        * The <b>schema_of_json()</b> function reads a <b>JSON String</b> directly from a <b>DataFrame</b> column and returns the <b>Schema</b> of that <b>JSON</b> string as a <b>DDL-Formatted String</b>.
    * <b>3</b>. <b>The Active Stream</b>: Then, that programmatically extracted <b>Schema</b> should be directly passed into the <b>Spark</b> function <b>from_json()</b> of the <b>Read Stream Query</b>.

In [0]:
# STEP 1: Read a Small Batch Sample Directly from the Live Kafka Topic
sample_batch_df = (spark.read
                        .format("kafka")
                        .option("kafka.bootstrap.servers", "<BROKER_IP>:<PORT>")
                        .option("subscribe", "topic_name")
                        .option("startingOffsets", "latest")  #To Grab the Fresh Messages
                        .load()
                        .limit(100) # Grab a Small Sample Size of 100 Messages
                        .select(col("value").cast("string").alias("json_string"))
                )

# STEP 2: Infer the Schema Using Pure DataFrame Expressions
# Grab the First Valid Row to Read the Schema String
schema_ddl = (sample_batch_df
                        .select(schema_of_json(col("json_string")))
                        .first()[0]
)

# STEP 3: Connect to the Kafka Topic and Launch the Real-Time Streaming Engine
streaming_df = (spark.readStream
                     .format("kafka")
                     .option("kafka.bootstrap.servers", "<BROKER_IP>:<PORT>")
                     .option("subscribe", "topic_name")
                     .load()
)

# STEP 4: Parse Message Values Seamlessly via the Final DDL-Formatted String Schema
parsed_stream_df = (streaming_df
                            .select(from_json(col("value").cast("string"), schema_ddl).alias("data"))
                            .select("data.*")
)

## Does the Above Code Works for Serverless Compute in Databricks?
* Yes. The above code is optimized for use in <b>Serverless Compute</b> as well, because of the below reasons -
    * <b>Zero RDD Overhead</b>: In <b>Databricks Serverless Compute</b>, the <b>RDD API</b>s are heavily restricted or outright disabled because the <b>Serverless Optimizer</b> prevents the <b>Lower Level API</b>s from accessing the <b>Serverless Cluster</b>.
    <br>To infer the <b>Schema</b> using modern, pure <b>DataFrame API</b>s without dropping down to <b>RDD</b>s or using <b>.collect()</b>, the built-in <b>Spark</b> function <b>schema_of_json()</b> can be used.
        * The built-in <b>Spark</b> function <b>schema_of_json()</b> is a highly optimized <b>Catalyst Optimizer</b> expression that is executed natively in <b>C++</b>, if the <b>Photon</b> engine if enabled, without the <b>JVM</b>, or, <b>Python</b> context switching.
    * <b>No Full Driver Serialization</b>: Unlike <b>.collect()</b>, which pulls the entire dataset to the <b>Driver</b>, the <b>DataFrame</b> function <b>.first()</b> only pulls one row containing the final inferred <b>DDL-Formatted String Schema</b>.
    * <b>Native DDL Support</b>: The built-in <b>Spark</b> functio <b>from_json()</b> accepts a standard <b>DDL String</b>, like <b>`STRUCT<id: INT, name: STRING>`</b> just as easily as it accepts a <b>StructType</b> object.

## Challenges of Programmatically Inferring Schema via Batch Samples
* While this approach avoids hardcoding the <b>Schema</b>, but, using this aproach generates the below production risks -
    * <b>Startup Latency</b>: Every time the <b>Streaming Workflow</b> restarts, the query for reading the sample batch of messages must run first, which delays the initialization of <b>Read Stream</b> query.
    * <b>Partial Sampling Risks</b>: If the sample of 100 messages does not contain optional, or, rare fields,then those fields will be missing from the inferred <b>Schema</b>.
    <br>Hence, <b>Spark</b> will drop any <b>Streaming</b> message that arrives later with those optional, or, rare fields.
    * <b>Schema Evolution Failures</b>: If an upstream application changes the structure of the <b>Schema</b> while the <b>Read Stream</b> is actively running in <b>Databricks</b>, then <b>Spark</b> will not be able to adapt this changed pattern dynamically.
    <br>The <b>Read Stream</b> will continue using the old <b>Schema</b> that was generated during startup, and, will populate <b>NULL</b> values in the newly added, or, updated fields.

## Fail-Safe Schema Infer When Kafka Topic is Empty or Kafka Topic Has Varying Payload Structures
* Using the plain built-in <b>Spark</b> function <b>schema_of_json()</b> alone exposes a major risk that it only evaluates a single row.
    * If <b>message #1</b> has fields <b>A</b>, and, <b>B</b>, but <b>message #1</b> has fields <b>C</b>, and, <b>D</b>, then the fields from <b>message #1</b> will be entirely skipped or dropped.
* To handle this cleanly in <b>Databricks Serverless Compute</b>, the built-in <b>Spark</b> function <b>schema_of_json_agg()</b> is used.
    * This function scans all sampled rows, safely combines varying payload keys, resolves the conflicts using the least common data type, and produces a single unified <b>DDL-Formatted String Schema</b>.

In [0]:
# STEP 1: Define a Hardcoded, Baseline Fallback DDL-Formatted String Schema. 
# This Prevents the Stream Initialization from Crashing if the Kafka Topic, or, Kinesis Topic is Completely Empty.
DEFAULT_FALLBACK_DDL = "STRUCT<id: INT, name: STRING, raw_payload: STRING>"

# STEP 2: Read a Small Batch Sample Directly from the Live Kafka Topic
sample_batch_df = (spark.read
                        .format("kafka")
                        .option("kafka.bootstrap.servers", "<BROKER_IP>:<PORT>")
                        .option("subscribe", "topic_name")
                        .option("startingOffsets", "latest")  #To Grab the Fresh Messages
                        .load()
                        .limit(100) # Grab a Small Sample Size of 100 Messages
                        .select(col("value").cast("string").alias("json_string"))
                )

try:
    # STEP 3: Aggregate All the Rows Across the 100 Sample Limit.
    # "schema_of_json_agg" Merges the Unique Attributes Across Varying Records Natively.
    aggregated_schema_row = (sample_batch_df
                                        .select(schema_of_json_agg(col("json_string")))
                                        .first()
    )

    # STEP 4: Extract the DDL-Formatted String Schema Out of the "Row" Object, Handling the Empty Topics Gracefully
    if aggregated_schema_row and aggregated_schema_row[0] is not None:
        schema_ddl = aggregated_schema_row[0]
        print(f"✅ Successfully inferred combined schema: {schema_ddl}")
    else:
        schema_ddl = DEFAULT_FALLBACK_DDL
        print(f"⚠️ Topic was empty during initialization. Falling back to default: {schema_ddl}")
except Exception as e:
    # Catch Any Unexpected Batch Read Issues or Malformed Data Errors
    schema_ddl = DEFAULT_FALLBACK_DDL
    print(f"❌ Error encountered during dynamic inference: {str(e)}. Falling back to safe layout.")

# STEP 5: Connect to the Kafka Topic and Launch the Real-Time Streaming Engine
streaming_df = (spark.readStream
                    .format("kafka")
                    .option("kafka.bootstrap.servers", "<BROKER_IP>:<PORT>")
                    .option("subscribe", "topic_name")
                    .load()
)

# STEP 6: Parse Message Values Seamlessly via the Final Unified DDL-Formatted String Schema
parsed_stream_df = (streaming_df
                            .select(from_json(col("value").cast("string"), schema_ddl).alias("data"))
                            .select("data.*")
)

## How the Above Code Fits Serverless Restrictions
* <b>schema_of_json_agg()</b>: The built-in <b>Spark</b> function <b>schema_of_json_agg()</b> works natively inside <b>Databricks Catalyst Optimizer</b> and <b>Photon</b> computation engines.
<br>It eliminates complex loop pipelines, or, <b>Python-to-JVM Serialization</b> layers.
* <b>Smart Data-Type Merging</b>: If <b>Row #1</b> contains an integer (<b>"age": 30</b>), and, <b>Rrow #2</b> contains a decimal float (<b>"age": 30.5</b>), the built-in <b>Spark</b> function <b>schema_of_json_agg()</b> merges those fields into a single <b>DOUBLE</b> representation automatically.
* <b>Safe .first() Check</b>: Checking <b>aggregated_schema_row[0] is not None</b> guarantees that the <b>Read Stream</b> code safely transitions to the baseline layout if no records are fetched from <b>Kafka Topic</b>.

# Different Types of Trigger in Spark Structured Streaming
* 

# How to Handle Poorly Formed, or, Malformed JSON Inputs During Streaming
* When <b>JSON</b> is parsed inside a <b>Read Stream</b> using the built-in <b>Spark</b> function <b>from_json()</b>, the default behavior for handling bad data, like missing commas, or, truncated payloads is <b>PERMISSIVE</b> mode.
<br><b>Spark</b> simply converts the broken attributes to <b>NULL</b> silently without failing the <b>Streaming</b> query.
* To explicitly catch these malformed payloads for a <b>Dead Letter Queue</b>, i.e, <b>DLQ</b>, without crashing the <b>Real Time Serverless Cluster</b> in <b>Databricks</b>, the <b>columnNameOfCorruptRecord</b> option must be combined with a slight modification to the dynamic <b>DDL-Formatted String Schema</b>.
* The below code leverages the built-in <b>Spark</b> function <b>schema_of_json_agg()</b> to infer the healthy attributes, dynamically appends a custom <b>bad_record</b> column, and splits the <b>Stream</b> into <b>Clean Data</b> and a <b>Dead Letter Queue</b>, i.e., <b>DLQ</b>.

In [0]:
# STEP 1: Define a Hardcoded, Baseline Fallback DDL-Formatted String Schema. 
# This Prevents the Stream Initialization from Crashing if the Kafka Topic, or, Kinesis Topic is Completely Empty.
DEFAULT_FALLBACK_DDL = "STRUCT<id: INT, name: STRING, raw_payload: STRING>"

# STEP 2: Read a Small Batch Sample Directly from the Live Kafka Topic
sample_batch_df = (spark.read
                        .format("kafka")
                        .option("kafka.bootstrap.servers", "<BROKER_IP>:<PORT>")
                        .option("subscribe", "topic_name")
                        .option("startingOffsets", "latest")  #To Grab the Fresh Messages
                        .load()
                        .limit(100) # Grab a Small Sample Size of 100 Messages
                        .select(col("value").cast("string").alias("json_string"))
                )

try:
    # STEP 3: Aggregate All the Rows Across the 100 Sample Limit.
    # "schema_of_json_agg" Merges the Unique Attributes Across Varying Records Natively.
    aggregated_schema_row = (sample_batch_df
                                        .select(schema_of_json_agg(col("json_string")))
                                        .first()
    )

    # STEP 4: Extract the DDL-Formatted String Schema Out of the "Row" Object, Handling the Empty Topics Gracefully
    if aggregated_schema_row and aggregated_schema_row[0] is not None:
        schema_ddl = aggregated_schema_row[0]
        print(f"✅ Successfully inferred combined schema: {schema_ddl}")
    else:
        schema_ddl = DEFAULT_FALLBACK_DDL
        print(f"⚠️ Topic was empty during initialization. Falling back to default: {schema_ddl}")
except Exception as e:
    # Catch Any Unexpected Batch Read Issues or Malformed Data Errors
    schema_ddl = DEFAULT_FALLBACK_DDL
    print(f"❌ Error encountered during dynamic inference: {str(e)}. Falling back to safe layout.")

# STEP 5: ⚡ CRITICAL STEP: Modify the Dynamic DDL-Formatted String Schem to Append the Corrupt Column.
# This Alters the "DEFAULT_FALLBACK_DDL" from "STRUCT<id: INT, name: STRING, raw_payload: STRING> to "STRUCT<id: INT, name: STRING, raw_payload: STRING, bad_record:STRING> safely.
corrupt_col_name = "bad_record"
final_streaming_schema = schema_ddl.replace(">", f", {corrupt_col_name}: STRING>")
print(f"Schema deployed to parser: {final_streaming_schema}")

# STEP 6: Connect to the Kafka Topic and Launch the Real-Time Streaming Engine
streaming_df = (spark.readStream
                    .format("kafka")
                    .option("kafka.bootstrap.servers", "<BROKER_IP>:<PORT>")
                    .option("subscribe", "topic_name")
                    .load()
)

# STEP 7: Parse Message Values Seamlessly via the Final Unified DDL-Formatted String Schema and Permissive Corrupt Options
parsed_stream_df = (streaming_df
                            .withColumn("data", from_json(
                                col("value").cast("string"), 
                                schema = final_streaming_schema,
                                options = {"columnNameOfCorruptRecord": corrupt_col_name}
                                )
                            )
                            .select("data.*")
)

# STEP 8: Create Separate DataFrames for "Clean Data", and, "Corrupt Data" Safely
# STEP 8.1: If "bad_record" is "NULL", then the Message is Parsed Perfectly.
clean_stream_df = parsed_stream_df.filter(col(corrupt_col_name).isNull()).drop(corrupt_col_name)
# STEP 8.2: If "bad_record" is "NOT NULL", then the Message is Broken.
dlq_stream_df = parsed_stream_df.filter(col(corrupt_col_name).isNotNull()).select(corrupt_col_name)

# STEP 9: Write to Separate Delta Tables
# STEP 9.1: Write Clean Data to a Delta Table
clean_query = (clean_stream_df.writeStream
                              .format("delta")
                              .outputMode("append")
                              .option("checkpointLocation", "/Volumes/oc_catalog/oc_schema/oc_volume/checkpoint_locations/clean_data")
                              .toTable("oc_catalog.oc_schema.oc_clean_table")
)
# STEP 9.2: Write Corrupt Data to Another Delta Table
dlq_query = (dlq_stream_df.writeStream
                        .format("delta")
                        .outputMode("append")
                        .option("checkpointLocation", "/Volumes/oc_catalog/oc_schema/oc_volume/checkpoint_locations/dlq_data")
                        .toTable("oc_catalog.oc_schema.oc_dlq_table")
)

* <b>Valid JSON Record</b>: For valid <b>JSON</b> records, like - <b>{"id": 101, "name": "Emma"}</b>, the fields map normally.
<br>The column <b>bad_record</b> becomes <b>NULL</b>.
<br>The row routes directly into the <b>Clean Table</b>, i.e., <b>oc_clean_table</b>.
* <b>Malformed JSON Record</b>: For malformed <b>JSON</b> records, like - <b>{"id": 102, "name": "Emma"</b>, which is missing the ending bracket, i.e., <b>)</b>, the <b>id</b> and <b>name</b> fields evaluate to <b>NULL</b>.
<br>The column <b>bad_record</b> captures the full <b>String</b> primitive, i.e., <b>{"id": 102, "name": "Emma"</b>.
<br>The row does not go to the <b>Clean Table</b>, i.e., <b>oc_clean_table</b>, and, logs immediately to the <b>Dead Letter Queue Table</b>, i.e., <b>oc_dlq_table</b> for developers to debug.
* It must be ensured that the <b>Clean Table</b>, i.e., <b>oc_clean_table</b>, and, the <b>Dead Letter Queue Table</b>, i.e., <b>oc_dlq_table</b> always use entirely different <b>Checkpoint</b> directories so that their transactional logs do not collide.

# How to Schedule the Streaming Job of Reading Messages from Kafka Topic and Storing the Message Data into Delta Table Using Spark Structured Streaming in Databricks

## 1. Scheduled Micro-Batch Processing (Batch Mode)
* For <b>Batch Processing</b> on a <b>Schedule</b>, the trigger type option <b>availableNow = True</b> is used.
* This trigger type option reads all the available messages currently sitting in the <b>Kafka Topic</b> as a <b>Single Batch</b>, processes, and loads the processed data into the <b>Sink</b>, i.e., the <b>Delta Table</b>, and, then automatically shuts down the <b>Stream</b>.
* It is ideal for saving money on <b>Serverless Compute</b> when run on a <b>CRON Schedule</b>, e.g., <b>hourly</b>, or, <b>daily</b>.

In [0]:
# Write Clean Data to a Delta Table Using Single-Batch Trigger
clean_query = (clean_stream_df.writeStream
                              .format("delta")
                              .outputMode("append")
                              .trigger(availableNow = True)  # Processes Current Backlogs of Messages and Stops the Stream
                              .option("checkpointLocation", "/Volumes/oc_catalog/oc_schema/oc_volume/checkpoint_locations/clean_data")
                              .toTable("oc_catalog.oc_schema.oc_clean_table")
)
# Write Corrupt Data to Another Delta Table Using Single-Batch Trigger
dlq_query = (dlq_stream_df.writeStream
                        .format("delta")
                        .outputMode("append")
                        .trigger(availableNow = True)  # Processes Current Backlogs of Messages and Stops the Stream
                        .option("checkpointLocation", "/Volumes/oc_catalog/oc_schema/oc_volume/checkpoint_locations/dlq_data")
                        .toTable("oc_catalog.oc_schema.oc_dlq_table")
)

# Block the Driver Execution Until the Single-Batch Finishes Processing
clean_query.awaitTermination()
dlq_query.awaitTermination()

## 2. Continuous Micro-Batching Processing (Near Real-Time)
* For <b>Near Real-Time Streaming</b>, the trigger type option <b>processingTime = "seconds/minutes"</b> is used, which is the standard design pattern for <b>Near Real-Time Streaming</b> in <b>Serverless Compute</b>.
* It polls the <b>Kafka Topic</b> at a specific time interval, e.g., <b>Every 10 Seconds</b>, or, <b>Every 1 Minute</b>, to process fresh messages as they land in the <b>Kafka Topic</b>.
    * The trigger type option <b>continuouse = "milliseconds"</b> is generally avoided in <b>Serverless Compute</b> due to high idle costs, as this trigger type option runs continuously, having very small latency of millisecond only.

In [0]:
# Write Clean Data to a Delta Table Using Time-Interval Trigger
clean_query = (clean_stream_df.writeStream
                              .format("delta")
                              .outputMode("append")
                              .trigger(processingTime = "10 seconds")  # Evaluates and Processes New Messages in Every 10 Seconds
                              .option("checkpointLocation", "/Volumes/oc_catalog/oc_schema/oc_volume/checkpoint_locations/clean_data")
                              .toTable("oc_catalog.oc_schema.oc_clean_table")
)
# Write Corrupt Data to Another Delta Table Using Time-Interval Trigger
dlq_query = (dlq_stream_df.writeStream
                        .format("delta")
                        .outputMode("append")
                        .trigger(processingTime = "10 seconds")  # Evaluates and Processes New Messages in Every 10 Seconds
                        .option("checkpointLocation", "/Volumes/oc_catalog/oc_schema/oc_volume/checkpoint_locations/dlq_data")
                        .toTable("oc_catalog.oc_schema.oc_dlq_table")
)

#### Why "processingTime" is Preferred Over "continuous" in Serverless Compute?
* True <b>Continuous Processing</b> keeps the <b>Processing Threads</b> permanently awake, which consume maximum <b>Driver</b> resources.
* Using the trigger type <b>processingTime = "10 seconds"</b> allows the <b>Serverless Compute</b> backends to gracefully scale or throttle <b>Tasks</b> if incoming message volumes fluctuate.

# How to Tune a Slow-Running Streaming Workflow That Reads Messages from a Kafka Topic
* <b>Reason of Slowness</b>:
    * In a <b>Streaming Workflow</b> that reads messages from a <b>Kafka Topic</b> in the <b>Read Stream</b> query, if the <b>Write Stream</b> query is executed using any one of the trigger type option, and, it can be seen that only one <b>Task</b> is created in the <b>Spark Job</b> for the entire <b>Stream</b> in the <b>Spark UI</b>, then it must be because, the messages are being read from the <b>Kafka Topic</b> using the <b>Read Stream</b> query from only one <b>Partition</b>.
    * The <b>Kafka Topic</b> that contains the <b>assignments</b> messages, sometimes constitutes of only one <b>Partition</b>.
    * This is why <b>Spark Structured Streaming</b> is able to connect to the <b>Kafka Topic</b>, having the <b>assignments</b> messages, and, read the messages through one <b>Partition</b> only, and, is not able to achieve <b>Parallelism</b>.
* <b>Tune the Slow-Running Streaming Workflow That Reads Messages from a Kafka Topic</b>:
    * If the number of <b>Partitions</b> is increased in the <b>Kafka Topic</b>, having the <b>assignments</b> messages, from <b>1</b> to any higher number, say <b>8</b>, then <b>Spark Structured Streaming</b> will be able to connect, and, read the messages through all the <b>8 Partitions</b> of the <b>Kafka Topic</b> in parallel.
    * Hence, if the <b>Write Stream</b> query is executed using any one of the trigger type option, then it can be seen that <b>8 Tasks</b> are created for the entire <b>Stream</b> in the <b>Spark UI</b>.
* <b>Summary</b>:
    * If a <b>Streaming Workflow</b> that reads messages from a <b>Kafka Topic</b> is running slowly, then one way to solve this issue is by increasing the number of <b>Partitions</b> in that <b>Kafka Topic</b>, so that the <b>Spark Structured Streaming</b> can read the messages in parallel fashion from multiple <b>Partitions</b>, and, process those messages with multiple <b>Tasks</b>.
    <br>This makes the <b>Streaming Workflow</b> runs faster.